In [62]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_stx"


In [63]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 12 firm-variable mappings across 10 firms.


firm_id,variable,notes,final_choice,num_candidates
NOVOb.CO,XSGA_COMPONENTS,"Candidate list misses 'Property management', which is a clearly populated SG&A overhead component above non-operating items. Because the explicit SG&A row exists but is mostly empty, included it plus populated overhead components for coverage across years. Excluded depreciation and direct cost rows. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Property management","Income Statement::Selling, General & Administrative Expenses - Other; Income Statement::Property management; Income Statement::Central administration",9
AXFO.ST,BE,Candidate list lacks a clear parent/owners' equity line. 'SubTotal' appears numerically to represent equity excluding non-controlling interests because it sits immediately before 'Non-controlling interests' and approximately sums with that row to 'Total equity'. Manual review recommended due to the ambiguous label. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] SubTotal,Balance Sheet::SubTotal,1
B3.ST,BE,"Preferred owner/parent equity line exists and should be used instead of Total Equity to avoid double counting with minority interest, but it is missing from the candidate list. Manual review flagged because final choice is outside candidates. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Eget kapital hänförligt till moderbolagets ägare",Balance Sheet::Eget kapital hänförligt till moderbolagets ägare,4
B3.ST,XSGA_COMPONENTS,"No explicit SG&A total row is present. Best overhead components are Research and development expenses, Other operating expenses, and Personal expenses. 'Personal expenses' is clearly the major overhead bucket but is missing from the candidate list, so this requires manual review. Depreciation/amortization excluded per instructions. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Personal expenses",Income Statement::Research and development expenses; Income Statement::Other operating expenses; Income Statement::Personal expenses,5
BALCO.ST,XSGA_COMPONENTS,"Candidate list clearly misses the important SG&A component 'Sales costs', which appears in the income statement and should be included along with 'Administration costs' and 'Other operating expenses'. No explicit single SG&A total row is present, so selected the main overhead buckets. Manual review flagged because one selected row was outside the provided candidate list. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Sales costs",Income Statement::Sales costs; Income Statement::Administration costs; Income Statement::Other operating expenses,5
BESQAB.ST,REVT,"Primary choice is candidate-list row 'Total Revenue'. Added 'Net sales' from the full preview as a fallback/prioritized alternate because it is the core sales label and may better represent top-line in some years; this row was missing from candidates, so manual review is required. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Net sales",Income Statement::Total Revenue; Income Statement::Net sales,6
BETSb.ST,BE,"Balance Sheet preview is not available here, so a proper book equity line cannot be identified. The candidate labels are income/cash-flow items, not balance-sheet equity.",,3
BETSb.ST,MIB,"Balance Sheet preview is not available here, so balance-sheet non-controlling interest cannot be mapped reliably. The available candidates are not balance-sheet MIB rows.",,3
BICO.ST,XSGA_COMPONENTS,"Candidate list appears incomplete because a clear SG&A component row 'Sales expenses' exists in the preview but is not in candidates. Per rules, selected it from full preview and set manual review. Included Administration expenses and Sales expenses as core SG&A components. Included Research and development expenses as separately reported overhead so XSGA-XRD can exclude it. Added Other external expenses and Other Operating Expenses to

In [64]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

XSGA_COMPONENTS manual reviews: 9 rows across 9 firms.


firm_id         variable  \
0  NOVOb.CO  XSGA_COMPONENTS   
1  NSISb.CO  XSGA_COMPONENTS   
2  PAALb.CO  XSGA_COMPONENTS   
3  RBREW.CO  XSGA_COMPONENTS   
4  ROCKb.CO  XSGA_COMPONENTS   
5  SKAKO.CO  XSGA_COMPONENTS   
6   TRYG.CO  XSGA_COMPONENTS   
7    UIE.CO  XSGA_COMPONENTS   
8    VWS.CO  XSGA_COMPONENTS   

                                               notes  \
0  Candidate list misses the best explicit SG&A r...   
1  Candidate list misses the important overhead r...   
2  Explicit SG&A row exists but appears unpopulat...   
3  Candidate list missed the clearly relevant row...   
4  No explicit SG&A row is present. Selected the ...   
5  Candidate list misses the clearly relevant sel...   
6  'Administration expenses' is the clearest SG&A...   
7  Candidate list only included 'Staff Costs', bu...   
8  Candidate list clearly misses the important SG...   

                                        final_choice  num_candidates  
0  Income Statement::Sales and distribution costs...              14  
1  Income Statement::Sales and distribution expen...               3  
2  Income Statement::Selling, General & Admin; In...               6  
3  Income Statement::Other Sales and distribution...               9  
4  Income Statement::Employee benefits expenses; ...               5  
5  Income Statement::Selling, General & Admin; In...              10  
6  Income Statement::Administration expenses; Inc...               3  
7  Income Statement::Staff Costs; Income Statemen...               2  
8  Income Statement::Distribution costs; Income S...              17